# 02 — Experiment Analysis

A full experiment readout, in the order a real experimentation platform
would run it:

1. **Trust check first** (Sample Ratio Mismatch) — before reading any
   result, verify the assignment mechanism delivered the designed split.
2. **Named comparisons** mirroring the source case study's 4 Key Insights,
   with 95% CIs and FDR correction.
3. **Direction concordance vs. ground truth** — a diagnostic only a
   simulation can run: which non-significant cells are power problems,
   and which are correctly null?
4. **Power & MDE table** — how big should this experiment have been, and
   what was it ever capable of detecting at its actual size?
5. **Adequately-powered rerun** — the same DGP at the sample size the
   power analysis prescribes, closing the loop on the diagnosis.

See `experiment_analysis.py` for full docstrings.

In [1]:
import os
import sys
from pathlib import Path

# Anchor all relative paths to THIS file's location, not the caller's cwd —
# so the notebook works whether run via Jupyter (kernel cwd = notebooks/) or
# as a script from anywhere (e.g. `python notebooks/02_....py` from repo root).
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:  # __file__ undefined inside a notebook kernel
    NOTEBOOK_DIR = Path.cwd()
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR.parent))

import pandas as pd

from data_generation import VoucherDGP
from experiment_analysis import (
    run_named_comparisons,
    fit_interaction_models,
    srm_check,
    power_mde_table,
    direction_concordance,
    NAMED_COMPARISONS,
    TIERS_ASC,
)

experiment_log = pd.read_csv("../data/processed/experiment_log.csv")
dgp = VoucherDGP(calibration_path="../data/raw_benchmarks/case_summary_tables.csv")

## 1. Trust check: Sample Ratio Mismatch (SRM)

Chi-square test per tier against the designed uniform 1/7 allocation,
with the industry-standard strict alarm threshold (p < 0.001). A failed
SRM check invalidates every downstream number — which is why it runs
BEFORE any result is read, not after.

In [2]:
srm = srm_check(experiment_log)
srm

,tier,n_users,n_conditions,chi2,p_value,srm_alarm
0,0-29,83308,7,8.77385,0.18670,False
1,30-69,36988,7,3.00984,0.80761,False
2,70-79,17302,7,7.48122,0.27862,False
3,80-89,15700,7,4.87185,0.56035,False
4,90-99,7598,7,4.04422,0.67069,False
5,100,39104,7,5.46041,0.48625,False


In [3]:
assert not srm["srm_alarm"].any(), "SRM detected — do not read results until assignment is debugged"
print("SRM check passed for all tiers — safe to read results.")

SRM check passed for all tiers — safe to read results.


## 2. Named comparisons (mirrors the source case study's 4 Key Insights)

Two-proportion z-tests (order rate) and Welch t-tests (profit/user) per
tier, now with 95% CIs on the absolute difference, BH-FDR corrected
within each metric family.

In [4]:
comparisons = run_named_comparisons(experiment_log)
comparisons.to_csv("../outputs/named_comparisons.csv", index=False)
comparisons[
    ["comparison", "tier", "metric", "abs_diff", "ci_low", "ci_high", "pct_lift", "q_value", "significant"]
]

,comparison,tier,metric,abs_diff,ci_low,ci_high,pct_lift,q_value,significant
0,voucher_vs_none,0-29,order_rate,0.00281,-0.00416,0.00979,3.49602,0.98064,False
1,voucher_vs_none,0-29,profit_per_user,0.00785,-0.00648,0.02218,5.10436,0.56557,False
2,voucher_vs_none,30-69,order_rate,-0.00547,-0.01695,0.00599,-5.34415,0.98064,False
3,voucher_vs_none,30-69,profit_per_user,-0.02503,-0.04380,-0.00625,-15.16982,0.17560,False
4,voucher_vs_none,70-79,order_rate,0.02134,0.00554,0.03723,27.07146,0.19365,False
5,voucher_vs_none,70-79,profit_per_user,0.00385,-0.01391,0.02161,4.16459,0.89478,False
6,voucher_vs_none,80-89,order_rate,0.01436,-0.00269,0.03145,16.50106,0.58994,False
7,voucher_vs_none,80-89,profit_per_user,-0.00096,-0.01571,0.01379,-1.25217,0.97250,False
8,voucher_vs_none,90-99,order_rate,0.02693,0.00104,0.05299,29.59787,0.32933,False
9,voucher_vs_none,90-99,profit_per_user,0.01066,-0.00504,0.02636,19.77420,0.54987,False


**Headline: 0 of 48 comparisons are significant after FDR correction.**
The CI columns make the reason visible without any p-value: nearly every
interval spans zero. The next two sections separate the two very
different explanations hiding inside that headline number.

## 3. Direction concordance vs. ground truth

Because the true effects were calibrated into the DGP, every comparison
can be classified: does the observed sign match the true sign, or was it
flipped by sampling noise? And critically — which cells have a
*negligible* true effect, where non-significance is the CORRECT outcome
rather than a power failure?

In [5]:
concordance = direction_concordance(comparisons, dgp)
concordance["true_effect"].value_counts()

true_effect
positive      27
negative      16
negligible     5
Name: count, dtype: int64

In [6]:
non_negligible = concordance[concordance["true_effect"] != "negligible"]
agreement_rate = non_negligible["sign_agrees"].mean()
print(
    f"Sign agreement among non-negligible true effects: "
    f"{agreement_rate:.0%} ({int(non_negligible['sign_agrees'].sum())}/{len(non_negligible)})"
)
concordance[concordance["sign_agrees"] == False]  # noqa: E712 — the sign-flipped cells

Sign agreement among non-negligible true effects: 70% (30/43)


,comparison,tier,metric,true_diff,observed_diff,true_effect,sign_agrees
2,voucher_vs_none,30-69,order_rate,0.00300,-0.00547,positive,False
5,voucher_vs_none,70-79,profit_per_user,-0.00300,0.00384,negative,False
7,voucher_vs_none,80-89,profit_per_user,0.00100,-0.00097,positive,False
12,minspend_299_vs_249,0-29,order_rate,0.00100,-0.00116,positive,False
22,minspend_299_vs_249,100,order_rate,-0.00100,0.00025,negative,False
25,copay_9_vs_0,0-29,profit_per_user,-0.00051,0.00019,negative,False
28,copay_9_vs_0,70-79,order_rate,-0.00116,0.00150,negative,False
30,copay_9_vs_0,80-89,order_rate,-0.00837,0.00396,negative,False
33,copay_9_vs_0,90-99,profit_per_user,-0.00140,0.00072,negative,False
36,count_3_vs_1,0-29,order_rate,0.00100,-0.00299,positive,False


Interpretation: most estimates point the right way, a substantial
minority are sign-flipped by noise — exactly the behavior expected from
an underpowered experiment, and a concrete warning against reading
individual cell directions at this sample size. The negligible-effect
cells are NOT failures: for them, "not significant" is the truth.

## 4. Power & MDE table (order-rate metric, Bonferroni-corrected)

Two complementary views per comparison x tier:
- **required_n_per_arm** — how big the experiment needed to be to detect
  the observed lift at 80% power.
- **mde_pct_lift** — the smallest lift detectable at the ACTUAL per-arm
  size (the standard experimentation-platform readout).

In [7]:
power_table = power_mde_table(comparisons, experiment_log)
power_table.to_csv("../outputs/power_mde_table.csv", index=False)
power_table

,comparison,tier,observed_pct_lift,current_n_per_arm,required_n_per_arm,mde_pct_lift,status
0,voucher_vs_none,0-29,3.5,11794,291900.0,17.9,underpowered
1,voucher_vs_none,30-69,-5.3,5185,92100.0,23.9,underpowered
2,voucher_vs_none,70-79,27.1,2466,5500.0,41.4,underpowered
3,voucher_vs_none,80-89,16.5,2258,12700.0,40.9,underpowered
4,voucher_vs_none,90-99,29.6,1060,4000.0,60.1,underpowered
5,voucher_vs_none,100,6.1,5650,250800.0,43.7,underpowered
6,minspend_299_vs_249,0-29,-1.4,11971,1724600.0,17.4,underpowered
7,minspend_299_vs_249,30-69,-0.4,5323,14963800.0,24.4,underpowered
8,minspend_299_vs_249,70-79,-5.5,2466,88900.0,35.9,underpowered
9,minspend_299_vs_249,80-89,-0.5,2210,10649500.0,37.7,underpowered


In [8]:
worked = power_table[
    (power_table["comparison"] == "voucher_vs_none") & (power_table["tier"] == "90-99")
].iloc[0]
print(
    f"Worked example — 90-99% tier, voucher_vs_none:\n"
    f"  observed lift {worked['observed_pct_lift']}% vs. MDE {worked['mde_pct_lift']}% "
    f"at n={worked['current_n_per_arm']:,}/arm\n"
    f"  -> the observed effect is ~half the smallest effect this cell could detect;\n"
    f"     required n/arm to detect it: {worked['required_n_per_arm']:,.0f}"
)

Worked example — 90-99% tier, voucher_vs_none:
  observed lift 29.6% vs. MDE 60.1% at n=1,060/arm
  -> the observed effect is ~half the smallest effect this cell could detect;
     required n/arm to detect it: 4,000


## 5. Adequately-powered rerun

The naive fix is scaling total N until the smallest key tier hits its
required per-arm size: the 90-99% tier is 3.7% of the population, so
4,000/arm x 7 arms / 3.7% ≈ **750,000 users**. (A real design would use
stratified oversampling of the small tiers to hit the same power at
roughly half that total — noted here, not implemented, since the point
of this section is closing the loop on the power diagnosis.)

In [9]:
big_log = dgp.simulate_users(n_users=750_000, seed=7)
big_comparisons = run_named_comparisons(big_log)

n_sig_small = int(comparisons["significant"].sum())
n_sig_big = int(big_comparisons["significant"].sum())
print(f"Significant after FDR at N=200k: {n_sig_small} / 48")
print(f"Significant after FDR at N=750k: {n_sig_big} / 48")

Significant after FDR at N=200k: 0 / 48
Significant after FDR at N=750k: 7 / 48


In [10]:
big_comparisons[big_comparisons["significant"]][
    ["comparison", "tier", "metric", "pct_lift", "q_value"]
]

,comparison,tier,metric,pct_lift,q_value
3,voucher_vs_none,30-69,profit_per_user,-9.44234,0.03141
4,voucher_vs_none,70-79,order_rate,18.75052,0.00287
10,voucher_vs_none,100,order_rate,16.62424,0.03698
22,minspend_299_vs_249,100,order_rate,-13.21206,0.03698
32,copay_9_vs_0,90-99,order_rate,-16.74668,0.03698
43,count_3_vs_1,80-89,profit_per_user,21.94355,0.00282
45,count_3_vs_1,90-99,profit_per_user,23.69652,0.04889


Significance emerges exactly where the power analysis predicted —
in the cells with large true effects — while the negligible-true-effect
cells correctly stay null even at 3.75x the sample. This closes the
loop: the N=200k non-results were a power problem for the big effects
and the right answer for the null ones, and the experiment-design
takeaway is quantified (750k naive, or less with stratified
oversampling of the 90-99% tier).

## 6. Interaction regression (tier x condition), cross-check

In [11]:
order_summary, profit_summary = fit_interaction_models(experiment_log)
print("Order (logit) — top 10 terms by |coef|:")
order_summary.reindex(order_summary["coef"].abs().sort_values(ascending=False).index).head(10)

Order (logit) — top 10 terms by |coef|:


,term,coef,std_err,p_value
0,Intercept,-2.43605,0.03385,0.00000
5,C(tier)[T.100],-0.93859,0.08171,0.00000
38,C(tier)[T.70-79]:C(condition)[T.s0_m199_c1],0.41548,0.10921,0.00014
35,C(tier)[T.90-99]:C(condition)[T.s0_m299_c3],0.37999,0.14538,0.00895
13,C(tier)[T.70-79]:C(condition)[T.s19_m199_c3],0.35387,0.10962,0.00125
28,C(tier)[T.70-79]:C(condition)[T.s19_m199_c1],0.31018,0.10991,0.00477
1,C(tier)[T.30-69],0.26533,0.05696,0.00000
20,C(tier)[T.90-99]:C(condition)[T.s0_m249_c3],0.25191,0.14974,0.09249
23,C(tier)[T.70-79]:C(condition)[T.s9_m249_c3],0.23657,0.11034,0.03203
18,C(tier)[T.70-79]:C(condition)[T.s0_m249_c3],0.22559,0.11018,0.04061


In [12]:
print("Profit (OLS) — top 10 terms by |coef|:")
profit_summary.reindex(profit_summary["coef"].abs().sort_values(ascending=False).index).head(10)

Profit (OLS) — top 10 terms by |coef|:


,term,coef,std_err,p_value
0,Intercept,0.15385,0.00407,0.00000
5,C(tier)[T.100],-0.12250,0.00715,0.00000
4,C(tier)[T.90-99],-0.09992,0.01399,0.00000
3,C(tier)[T.80-89],-0.07693,0.01014,0.00000
2,C(tier)[T.70-79],-0.06146,0.00965,0.00000
17,C(tier)[T.30-69]:C(condition)[T.s0_m249_c3],-0.03288,0.01035,0.00149
35,C(tier)[T.90-99]:C(condition)[T.s0_m299_c3],0.02943,0.01959,0.13300
22,C(tier)[T.30-69]:C(condition)[T.s9_m249_c3],-0.02567,0.01036,0.01322
13,C(tier)[T.70-79]:C(condition)[T.s19_m199_c3],0.02417,0.01368,0.07737
28,C(tier)[T.70-79]:C(condition)[T.s19_m199_c1],0.02265,0.01375,0.09951
